In [1]:
from pathlib import Path

print("=== Available Model Artifacts ===")

for path in Path("/kaggle/input").rglob("*"):
    if path.is_file():
        print(path)

=== Available Model Artifacts ===
/kaggle/input/notebooks/marwanedrissi/00-kaggle-gpu-check/__notebook__.ipynb
/kaggle/input/notebooks/marwanedrissi/00-kaggle-gpu-check/__output__.json
/kaggle/input/notebooks/marwanedrissi/00-kaggle-gpu-check/__results__.html
/kaggle/input/notebooks/marwanedrissi/00-kaggle-gpu-check/custom.css
/kaggle/input/notebooks/marwanedrissi/00-kaggle-gpu-check/infra_artifacts/experiment_info.json
/kaggle/input/notebooks/marwanedrissi/00-kaggle-gpu-check/infra_artifacts/tiny_model_state_dict.pt


In [2]:
import torch
import torch.nn as nn
import json

from pathlib import Path

print("=== IndustrialDefect AI ===")
print("INFRA-07: Independent Model Restoration")

# 1. Retrouver les poids sauvegardés
model_files = list(
    Path("/kaggle/input").rglob("tiny_model_state_dict.pt")
)

assert len(model_files) == 1, (
    f"Expected exactly one model file, found {len(model_files)}"
)

model_path = model_files[0]
metadata_path = model_path.parent / "experiment_info.json"

print("\nModel path:", model_path)

# 2. Recréer l'architecture du modèle
def build_model():
    return nn.Sequential(
        nn.Linear(4, 16),
        nn.ReLU(),
        nn.Linear(16, 2)
    )

model = build_model()

# 3. Charger les poids sur CPU
state_dict = torch.load(
    model_path,
    map_location="cpu",
    weights_only=True
)

model.load_state_dict(state_dict)
model.eval()

# 4. Charger les métadonnées
with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("Experiment:", metadata["experiment"])
print("Original device:", metadata["device"])

# 5. Récupérer les données de référence
reference_input = torch.tensor(
    metadata["reference_input"],
    dtype=torch.float32
)

expected_output = torch.tensor(
    metadata["reference_output"],
    dtype=torch.float32
)

# 6. Effectuer une nouvelle prédiction
with torch.inference_mode():
    restored_output = model(reference_input)

# 7. Vérifier les résultats
is_valid = torch.allclose(
    restored_output,
    expected_output,
    atol=1e-5,
    rtol=1e-5
)

print("\n=== RESTORATION RESULTS ===")
print("Expected:", expected_output)
print("Restored:", restored_output)
print("Predictions match:", is_valid)
print("Execution device:", next(model.parameters()).device)

assert is_valid, "Model restoration failed!"

print("\nSUCCESS: Model restored without retraining!")

=== IndustrialDefect AI ===
INFRA-07: Independent Model Restoration

Model path: /kaggle/input/notebooks/marwanedrissi/00-kaggle-gpu-check/infra_artifacts/tiny_model_state_dict.pt
Experiment: INFRA-07
Original device: cuda

=== RESTORATION RESULTS ===
Expected: tensor([[-0.4784,  0.0699]])
Restored: tensor([[-0.4784,  0.0699]])
Predictions match: True
Execution device: cpu

SUCCESS: Model restored without retraining!
